# Beer Game: token constraints, information sharing and bullwhip

**Start here. This notebook implements the new `ThoughtsReflection_Paragraph.docx`, not the earlier training experiment.**

1. **H1:** short preparation budgets reduce the fitted weight on outstanding replenishment (beta).
2. **H2:** short preparation budgets increase team holding/backlog cost.
3. Both conditions share all four roles' current inventories. There is **no trained/control condition**.
4. The same GPT-5.4 nano snapshot uses `reasoning.effort="none"` in both conditions. The API enforces different **visible memo-token caps**, followed by an identical **192-token structured final answer**. This is an operational proxy for constrained preparation, not a measure of human seconds or total internal cognition.
5. **Currently prepared: API calibration 2 (paid).** To run a free software check, set `MODE="offline"`. API calibration comes before a small game pilot; no automatic large study.

The source document is [here](../data/source/ThoughtsReflection_Paragraph.docx). Full protocol: [docs/PROTOCOL.md](../docs/PROTOCOL.md).

**Calibration 1 completed:** 192 successful calls, estimated $0.0540. Its short 176-token allowance truncated 3/24 memos (12.5%), below the existing 20% requirement. Adequate 240 tokens had no truncation; the 480-token benchmark differed by a median of 2 order units. The game is not yet enabled.

**Next prepared run:** fixed 96 / 240 / 480 token screens, 144 calls, with the original reference screen preserved. Thresholds, model, final-answer allowance and game rules are unchanged. This is a calibration refinement, not a search for favorable H1/H2 results.


## 1. Setup (once)

Use a Python 3.9+ Jupyter kernel. If imports fail, run the commented installation line below, then restart the kernel. Never paste an API key into notebook source or outputs.

The setup and report cells install missing Word/Excel export packages into the active kernel environment. A report-export error does not require rerunning the simulation.

In [1]:
from pathlib import Path
import sys, os, getpass, importlib
from IPython.display import display, Markdown, Image
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for parent in candidates for p in
    (parent, parent / "beer_game_experiment", parent / "Project" / "beer_game_experiment")
    if (p / "src" / "beergame_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the beer_game_experiment project.")
# If needed, uncomment this line, run it, then restart the kernel:
# %pip install -r {ROOT / "requirements.txt"}
sys.path.insert(0, str(ROOT / "src"))
from beergame_study import Config, run, run_directory, report, software_checks
from beergame_study.api import StudyStopped
print("Project:", ROOT)

# Report exporters must be installed in THIS notebook's Python environment.
def ensure_report_dependencies():
    import importlib.util, subprocess
    packages = {"docx": "python-docx>=1,<2", "openpyxl": "openpyxl>=3,<4"}
    missing = [package for module, package in packages.items()
               if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing missing report packages into:", sys.executable)
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        importlib.invalidate_caches()
    return True

ensure_report_dependencies()


Project: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment


True

## 2. Settings and cost

- `MODE="offline"`: deterministic software check, **not GPT evidence**. The same synthetic rule operates in both conditions.
- `MODE="api"`, `STAGE="calibration"`: 24 independent hypothetical states × reference/short/adequate/doubled allowance × two stages = **192 planned calls**. P25/P95 of reference memo use suggest candidate caps; the screen checks truncation and benchmark differences. This economical screen differs from the reflection's four-chain human calibration.
- After a **passed API calibration**, set `STAGE="game"`, copy its suggested caps and folder into the settings. Start with `GROUPS_PER_CONDITION=1` (two chains, eight agents). Use **2 for a four-chain, 16-agent pilot**. The reflection's main design uses **10 per condition = 20 chains = 80 agents**; do not automatically scale.
- Game: 40 weeks, first four fixed-order warmup weeks, then 36 decision weeks. Another 12 weeks are scored automatically, with no API calls.
- Rates checked 2026-10-04: $0.20 input / $0.02 cached input / $1.25 output per million tokens. [Official model documentation](https://developers.openai.com/api/docs/models/gpt-5.4-nano). Cost estimates below assume 1,500 input tokens/call; actual cost varies with history and output. The local allowance is a guard, not your account balance.

**Resume:** keep experiment settings and replicate ID fixed; raise only the dollar allowance if required. Scientific/code changes create a new folder. No earlier experiment data are reused.

For the prepared second calibration, the reference phase is read from the earlier completed run and is not purchased again. The 96-token short cap is fixed in advance. This run uses 144 planned calls, with a separate saved folder. A passed first reference does not make the second screen pass automatically.

In [2]:
MODE = "api"                # Change to "api" only when ready to spend
STAGE = "calibration"           # "calibration" first, then "game"
GROUPS_PER_CONDITION = 1         # 1=8 agents, 2=16 agents, 10=80 agents (two conditions)
SHORT_TOKENS = 96               # Fixed in advance for calibration 2
ADEQUATE_TOKENS = 240           # Fixed in advance for calibration 2
CALIBRATION_METHOD = "fixed"    # Second screen; do not recompute the original percentiles
CALIBRATION_PARENT = str(ROOT / "data/runs/calibration_api_6778221cada613b6")
CALIBRATION_RUN = ""            # Paste passed API calibration folder for STAGE="game"
COLLECT_DEBRIEFS = False         # Optional post-game questionnaire; +1 call per agent
BUDGET_USD = 1.05               # Total allowance for THIS run, including cached spending
REPLICATE_ID = "token-calibration-02"
CONFIG = Config(mode=MODE, stage=STAGE, replicate_id=REPLICATE_ID,
    groups_per_condition=GROUPS_PER_CONDITION, short_tokens=SHORT_TOKENS,
    adequate_tokens=ADEQUATE_TOKENS, calibration_run=CALIBRATION_RUN,
    calibration_method=CALIBRATION_METHOD, calibration_parent=CALIBRATION_PARENT,
    budget_usd=BUDGET_USD, collect_debriefs=COLLECT_DEBRIEFS)
CONFIG.validate()
run_dir = run_directory(ROOT, CONFIG)
calibration_screens = 3 if CALIBRATION_METHOD == "fixed" else 4
planned = 48 * calibration_screens if STAGE == "calibration" else 8 * GROUPS_PER_CONDITION * (1 + 2 * 36 + int(COLLECT_DEBRIEFS))
output_allowance = (24 * ((1024 if CALIBRATION_METHOD == "percentile" else 0) + SHORT_TOKENS + ADEQUATE_TOKENS + 2*ADEQUATE_TOKENS + calibration_screens*192)
                    if STAGE == "calibration" else
                    GROUPS_PER_CONDITION * (4*36*(SHORT_TOKENS+ADEQUATE_TOKENS+2*192) + 8*192 + 8*384*int(COLLECT_DEBRIEFS)))
scenario = (planned*1500*.20 + output_allowance*1.25)/1e6
print(f"{MODE} / {STAGE}; {planned} planned calls plus transport retries/replacements")
print(f"Illustrative cost at 1,500 input tokens/call and full output use: ${scenario:.2f}; not a quote.")
print(f"Request pacing alone: at least {planned * CONFIG.request_interval_s / 60:.1f} minutes for an uncached run.")
print("Local allowance:", BUDGET_USD, "USD; result folder:", run_dir)

api / calibration; 144 planned calls plus transport retries/replacements
Illustrative cost at 1,500 input tokens/call and full output use: $0.08; not a quote.
Request pacing alone: at least 2.4 minutes for an uncached run.
Local allowance: 1.05 USD; result folder: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6f7902f51127105b


## 3. Free checks before any paid call

Checks include simulator conservation, final-order consequences, beta recovery with legitimate zero orders, and a sensible benchmark outperforming always-zero/always-large ordering. They verify the software, not GPT behavior.

In [3]:
checks = software_checks()
display(checks)
assert checks.passed.all()
print("Free checks passed. No API calls made by this cell.")

,check,passed,detail
0,Simple demand benchmark beats always-zero and ...,True,"{'zero': 9368.0, 'demand': 156.0, 'large': 262..."
1,Final order affects score: retailer,True,"[116.0, 0.0, 592.0]"
2,Final order affects score: wholesaler,True,"[168.0, 0.0, 352.0]"
3,Final order affects score: distributor,True,"[204.0, 0.0, 208.0]"
4,Final order affects score: factory,True,"[224.0, 0.0, 160.0]"
5,"Known beta recovery, including valid zero orders",True,0.65
6,Bullwhip ratio undefined for constant denominator,True,"NaN, not infinity or zero"


Free checks passed. No API calls made by this cell.


## 4. Run the selected stage

This is the only paid cell in API mode. It requests the key in a hidden prompt. Progress shows completed/remaining calls, cached calls, active requests and spending.

A preparation memo reaching its cap is **not a comprehension failure**: the same final-answer allowance follows. Final answers must have a valid nonnegative integer and correct state readbacks. Temporary service errors get bounded retries with pacing/cooldown. A failed final answer may replace the entire four-role group once; failed groups stay recorded and inferential p-values are withheld after replacement. There is no retry-until-good strategy and no automatic model/budget upgrade.

In [4]:
client = None
try:
    if MODE == "api":
        from openai import AsyncOpenAI
        key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
        if not key.strip():
            raise ValueError("No API key entered.")
        client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=CONFIG.timeout_s)
        del key
    run_dir = await run(ROOT, CONFIG, client=client)
    print("Stage complete. Read the report below before proceeding.")
except StudyStopped as exc:
    print("Stopped:", str(exc))
    print("Saved folder:", run_dir)
finally:
    if client is not None:
        await client.close()

Stage complete. Read the report below before proceeding.


## 5. Conclusions and full results

The report leads with conclusions. It saves Word, Markdown, Excel, CSV and plots inside the displayed run folder.

- **H1:** group beta from four identified role estimates; report missing/unreliable fits rather than forcing a value.
- **H2:** total team cost, analyzed independently of beta availability. No p-values from the tiny pilot or synthetic data.
- **Bullwhip (牛鞭效应):** order variance divided by incoming-demand variance at each role, plus order variance divided by external-customer-demand variance for comparison along the chain. Above 1 means amplification relative to that denominator. Also inspect the time series, backlog and inventory. Constant denominators produce missing ratios, not invented zeros.
- The warmup and demand step are retained in the variance window; after the step alone, customer demand is constant. Ratios from this transient experiment are descriptive, not stationary demand estimates.
- Zero orders may follow excess ordering/stock. There is **no advertised order maximum** and no reward for matching a cap. An undisclosed numerical safety check halts absurd-scale orders above 100,000 without clipping them.

[MIT rules and delay dynamics](https://web.mit.edu/jsterman/www/SDG/beergame.html) · [Original bullwhip research](https://pubsonline.informs.org/doi/10.1287/mnsc.43.4.546)

In [5]:
# This cell only reads saved results and writes reports; no simulation/API calls.
# Works even if the kernel was started before the setup cell gained its dependency check.
import importlib.util, subprocess, sys
missing = [package for module, package in
           {"docx": "python-docx>=1,<2", "openpyxl": "openpyxl>=3,<4"}.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

if (run_dir / "progress.json").exists():
    display(Markdown(report(run_dir)))
    for name in ("order_trajectories.png", "bullwhip.png"):
        if (run_dir / name).exists():
            display(Image(filename=str(run_dir / name)))
    print("Word report:", run_dir / "REPORT.docx")
    print("Detailed workbook:", run_dir / "DETAILS.xlsx")
    if (run_dir / "calibration_recommendation.json").exists():
        import json
        recommendation = json.loads((run_dir / "calibration_recommendation.json").read_text())
        print("Suggested settings (review before using):", recommendation)
        print("CALIBRATION_RUN =", repr(str(run_dir)))
else:
    print("No saved stage yet.")

# Token-budget Beer Game — conclusions

1. Real API responses; LLM simulation, not human evidence.
2. Run status: complete. Estimated cost/reservations: $0.0381.
3. Saved successful API stages: 144; other saved attempts: 0.
4. Suggested memo caps: 96 / 240 tokens; doubled benchmark 480.
5. Manipulation screen passed: True. This is a provisional feasibility check, not proof of a human time-pressure equivalent.
6. Review the state-level choices; then run one group per condition before scaling.
Calibration uses 24 independent hypothetical states (all four roles), not complete game chains. This follow-up uses prospectively fixed caps and retains the earlier completed reference screen. It does not reselect percentiles or change the pass thresholds.
Short truncation: 91.7% (required at least 20%); adequate truncation: 0.0% (required at most 5%).
Adequate versus doubled allowance: median absolute order difference 4.0 units (screen requires at most 4).

## Calibration summary

```
   screen  mean_tokens  truncated_fraction  mean_order
 adequate     151.1250              0.0000      8.9167
benchmark     192.4167              0.0000      6.2500
    short      95.3750              0.9167      6.3750
```

## Calibration state results

```
   screen  item        role  inventory  supply_line  order  plan_tokens  final_tokens  plan_truncated  plan_limit  latency_s
    short     0    retailer          0            0     12           96            39            True          96     3.8590
    short     1    retailer          0           12     16           96            39            True          96     1.5922
    short     2    retailer          0           36     20           96            39            True          96     1.5777
    short     3    retailer         24            0      6           87            39           False          96     2.2658
    short     4    retailer         24           12      0           96            39            True          96     1.9785
    short     5    retailer         24           36      0           96            39            True          96     1.7142
    short     6  wholesaler          0            0      8           96            39            True          96     1.7817
    short     7  wholesaler          0           12      4           96            39            True          96     2.3106
    short     8  wholesaler          0           36      0           96            39            True          96     2.9019
    short     9  wholesaler         24            0      0           96            39            True          96     1.9379
    short    10  wholesaler         24           12      8           96            39            True          96     1.9633
    short    11  wholesaler         24           36     20           96            39            True          96     1.9774
    short    12 distributor          0            0     16           96            39            True          96     1.6387
    short    13 distributor          0           12      8           96            39            True          96     1.8391
    short    14 distributor          0           36      0           96            39            True          96     2.8792
    short    15 distributor         24            0      5           96            39            True          96     2.1493
    short    16 distributor         24           12      6           96            39            True          96     2.0408
    short    17 distributor         24           36      4           90            39           False          96     1.7038
    short    18     factory          0            0      0           96            39            True          96     1.6886
    short    19     factory          0           12      0           96            39            True          96     2.1619
    short    20     factory          0           36     12           96            39            True          96     1.6033
    short    21     factory         24            0      0           96            39            True          96     1.8439
    short    22     factory         24           12      0           96            39            True          96     1.7094
    short    23     factory         24           36      8           96            39            True          96     2.1386
 adequate     0    retailer          0            0     16          212            39           False         240     2.2472
 adequate     1    retailer          0           12     20          175            39           False         240     1.9702
 adequate     2    retailer          0           36     16          167            39           False         240     2.0990
 adequate     3    retailer         24            0      4          168            39           False         240     2.6172
 adequate     4    retailer         24           12      0          148            39           False         240     2.2089
 adequate     5    retailer         24           36      0          119            39           False         240     1.8912
 adequate     6  wholesaler          0            0     12          133            39           False         240     1.9000
 adequate     7  wholesaler          0           12      0          139            39           False         240     2.3801
 adequate     8  wholesaler          0           36     20          114            39           False         240     2.6063
 adequate     9  wholesaler         24            0      8          139            39           False         240     2.9525
 adequate    10  wholesaler         24           12      8          142            39           False         240     2.2577
 adequate    11  wholesaler         24           36      4          117            39           False         240     1.9582
 adequate    12 distributor          0            0     16          128            39           False         240     1.8920
 adequate    13 distributor          0           12     12          132            39           False         240     2.4405
 adequate    14 distributor          0           36     12          174            39           False         240     2.0852
 adequate    15 distributor         24            0      4          133            39           False         240     1.7950
 adequate    16 distributor         24           12      6          175            39           False         240     2.5352
 adequate    17 distributor         24           36      8          161            39           False         240     2.1809
 adequate    18     factory          0            0     12          174            39           False         240     2.2997
 adequate    19     factory          0           12     20          148            39           False         240     1.8175
 adequate    20     factory          0           36      0          142            39           False         240     2.1070
 adequate    21     factory         24            0      0          147            39           False         240     2.1168
 adequate    22     factory         24           12      8          174            39           False         240     2.3845
 adequate    23     factory         24           36      8          166            39           False         240     2.0594
benchmark     0    retailer          0            0     12          187            39           False         480     2.1318
benchmark     1    retailer          0           12      0          214            39           False         480     2.2648
benchmark     2    retailer          0           36      8          204            39           False         480     2.2342
benchmark     3    retailer         24            0      0          137            39           False         480     2.0286
benchmark     4    retailer         24           12      4          165            39           False         480     1.9839
benchmark     5    retailer         24           36      8          224            39           False         480     2.1986
benchmark     6  wholesaler          0            0     20          207            39           False         480     2.1750
benchmark     7  wholesaler          0           12     20          240            39           False         480     2.5564
benchmark     8  wholesaler          0           36      0          166            39           False         480     2.3130
benchmark     9  wholesaler         24            0      8          182            39           False         480     2.6139
benchmark    10  wholesaler         24           12      0          168            39           False         480     2.0184
benchmark    11  wholesaler         24           36      4          131            39           False         480     4.5942
benchmark    12 distributor          0            0     16          209            39           False         480     1.9520
benchmark    13 distributor          0           12     16          217            39           False         480     2.2570
benchmark    14 distributor          0           36      8          201            39           False         480     2.6248
benchmark    15 distributor         24            0      0          167            39           False         480     2.1214
benchmark    16 distributor         24           12      6          213            39           False         480     3.3876
benchmark    17 distributor         24           36      0          214            39           False         480     2.3143
benchmark    18     factory          0            0      0          206            39           False         480     2.0532
benchmark    19     factory          0           12     12          190            39           False         480     2.0257
benchmark    20     factory          0           36      0          189            39           False         480     2.2128
benchmark    21     factory         24            0      0          186            39           False         480     2.2248
benchmark    22     factory         24           12      0          239            39           False         480     2.2541
benchmark    23     factory         24           36      8          162            39           False         480     2.3777
```

## Interpretation limits

This protocol restricts visible preparation tokens plus a common final-response allowance, with reasoning effort none. It does not directly limit hidden cognitive effort, wall-clock time or human thinking time. Prompts cannot reliably count remaining tokens; the API hard cap is the enforced constraint. A truncated memo is accepted as a deadline outcome, then a common 192-token stage solicits the order. The final stage can still involve implicit computation, and unused token allowance is not effort used.
Technical retries retain the same prompt and cap. Failed final responses can replace a whole group only within the configured limit; all failed attempts remain recorded. Replacement selection withholds inferential p-values. No order is clipped or replaced by a fabricated answer. The separate emergency scale check stops the run rather than making a boundary order.

Sources: [provided reflection](../../source/ThoughtsReflection_Paragraph.docx); [MIT Beer Game](https://web.mit.edu/jsterman/www/SDG/beergame.html); [Lee, Padmanabhan & Whang](https://pubsonline.informs.org/doi/10.1287/mnsc.43.4.546); [OpenAI model documentation](https://developers.openai.com/api/docs/models/gpt-5.4-nano).


Word report: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6f7902f51127105b/REPORT.docx
Detailed workbook: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6f7902f51127105b/DETAILS.xlsx
Suggested settings (review before using): {'short_tokens': 96, 'adequate_tokens': 240, 'benchmark_tokens': 480, 'median_absolute_adequate_benchmark_order_difference': 4.0, 'manipulation_screen_passed': True, 'mode': 'api', 'calibration_method': 'fixed', 'calibration_parent': '/Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6778221cada613b6', 'reference_screen_passed': True, 'note': 'Pilot thresholds, not validated psychometric cutoffs. No automatic main study.'}
CALIBRATION_RUN = '/Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6f7902f51127105b'


## 6. What to do next

1. Run this entire notebook in **offline calibration** mode once.
2. Run **API calibration**; review whether the cap binds and the generous/doubled decisions are reasonably stable. A failed screen means the token contrast is not yet established; do not call it a successful time-pressure manipulation.
3. For a passed screen, copy its two budgets and folder into the settings, change `STAGE="game"`, and run **one group per condition** first. Keep `REPLICATE_ID` fixed to resume.
4. Assess beta identification, total costs, actual token use, failure rates and bullwhip patterns. Plan group-level replication/power before the proposed **10 groups per condition**. The previous $3 budget is not assumed sufficient for 80 agents.

No prompt, token budget or model guarantees realistic human behavior, bullwhip amplification or significant H1/H2. The study tests those outcomes.

The updated reflection also proposes explanations/questionnaires. Set `COLLECT_DEBRIEFS=True` **before a new game run** to save one concise retrospective self-report per agent. This is optional, adds API cost, and is separate from decision data. Model self-reports do not establish the actual mechanism behind decisions.